In [0]:
taxes_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__taxes/"
)

taxes_df = (
    spark.read
    .option("multiLine", "false")
    .json(taxes_path + "*.jsonl.gz")
)

print("Total rows:", taxes_df.count())
print("Total columns:", len(taxes_df.columns))

taxes_df.printSchema()

In [0]:
# =====================================================
# Tax cardinality per listing
# =====================================================

from pyspark.sql import functions as F

tax_counts = (
    taxes_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("tax_count")
    )
)

display(
    tax_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("tax_count").alias("avg_taxes"),
        F.expr("percentile_approx(tax_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(tax_count, 0.50)").alias("median"),
        F.expr("percentile_approx(tax_count, 0.75)").alias("p75"),
        F.min("tax_count").alias("min_taxes"),
        F.max("tax_count").alias("max_taxes")
    )
)

In [0]:
# =====================================================
# Validate taxes → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

tax_parent_ids = (
    taxes_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    tax_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    tax_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Tax parent IDs:", tax_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# TAX DATA QUALITY — NULL / COMPLETENESS PROFILE
# =====================================================

total_rows = taxes_df.count()

quality_profile = taxes_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in taxes_df.columns
]).first().asDict()

quality_data = [
    (
        column,
        int(quality_profile[column]),
        round(
            (1 - quality_profile[column] / total_rows) * 100,
            2
        )
    )
    for column in taxes_df.columns
]

quality_df = spark.createDataFrame(
    quality_data,
    ["column_name", "null_count", "completeness_pct"]
)

display(
    quality_df
    .orderBy(F.asc("completeness_pct"))
)

In [0]:
# =====================================================
# TAX DATA QUALITY — BLANK STRING CHECK
# =====================================================

string_columns = [
    field.name
    for field in taxes_df.schema.fields
    if field.dataType.simpleString() == "string"
]

blank_profile = taxes_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

blank_data = [
    (column, int(blank_counts[column]))
    for column in string_columns
    if blank_counts[column] > 0
]

if blank_data:
    display(
        spark.createDataFrame(
            blank_data,
            ["column_name", "blank_count"]
        )
        .orderBy(F.desc("blank_count"))
    )
else:
    print("No empty or whitespace-only string values found.")

In [0]:
# =====================================================
# TAX PERCENTAGE — VALIDITY CHECK
# =====================================================

tax_pct = (
    taxes_df
    .filter(F.col("tax_percentage").isNotNull())
    .select(
        F.col("tax_percentage").cast("double").alias("tax_percentage")
    )
)

tax_pct.agg(
    F.count("*").alias("non_null_count"),
    F.sum(
        F.when(F.col("tax_percentage").isNull(), 1).otherwise(0)
    ).alias("invalid_numeric_values"),
    F.min("tax_percentage").alias("min_tax_percentage"),
    F.max("tax_percentage").alias("max_tax_percentage")
).show()

In [0]:
# =====================================================
# TAX PERCENTAGE — PROPER NUMERIC VALIDATION
# =====================================================

tax_pct_quality = taxes_df.filter(
    F.col("tax_percentage").isNotNull()
).select(
    "tax_percentage",
    F.col("tax_percentage").cast("double").alias("tax_percentage_numeric")
)

tax_pct_quality.agg(
    F.count("*").alias("non_null_raw_values"),
    F.sum(
        F.when(
            F.col("tax_percentage_numeric").isNull(),
            1
        ).otherwise(0)
    ).alias("non_numeric_values"),
    F.sum(
        F.when(
            F.col("tax_percentage_numeric") < 0,
            1
        ).otherwise(0)
    ).alias("negative_values"),
    F.sum(
        F.when(
            F.col("tax_percentage_numeric") > 100,
            1
        ).otherwise(0)
    ).alias("above_100_values")
).show()

In [0]:
# =====================================================
# TAX TYPE DISTRIBUTION
# =====================================================

display(
    taxes_df
    .groupBy("tax_type")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy(F.desc("record_count"))
)

# =====================================================
# JURISDICTION TYPE DISTRIBUTION
# =====================================================

display(
    taxes_df
    .groupBy("tax_jurisdiction__region__region_type")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy(F.desc("record_count"))
)

In [0]:
# =====================================================
# TAX JURISDICTION COMPLETENESS BY TAX TYPE
# =====================================================

display(
    taxes_df
    .groupBy("tax_type")
    .agg(
        F.count("*").alias("record_count"),

        F.sum(
            F.when(
                F.col("tax_jurisdiction__region__region_name").isNull(),
                1
            ).otherwise(0)
        ).alias("region_name_null"),

        F.sum(
            F.when(
                F.col("tax_jurisdiction__region__region_type").isNull(),
                1
            ).otherwise(0)
        ).alias("region_type_null"),

        F.sum(
            F.when(
                F.col("tax_jurisdiction__tax_jurisdiction_id").isNull(),
                1
            ).otherwise(0)
        ).alias("jurisdiction_id_null")
    )
    .orderBy("tax_type")
)

# Taxes Profile — Complete Profiling Summary

## 1. Structure

- **Total records:** 3.6M+
- **Columns:** 11
- **Grain:** Item listing × tax jurisdiction/type

The `taxes_df` dataset is a **one-to-many child of Item Details**. A single listing can have multiple tax records, typically representing different tax jurisdictions or tax-related entries.

### Tax Cardinality per Listing

| Metric | Result |
|---|---:|
| Parent listings | 72,784 |
| Average tax records | 47.05 |
| P25 | 47 |
| Median | 47 |
| P75 | 47 |
| Minimum | 47 |
| Maximum | 57 |

The normal pattern is therefore approximately:

```text
Item Details listing
        │
        ├── Tax record
        ├── Tax record
        ├── Tax record
        ├── ...
        └── ~47 tax records
```

This confirms that taxes_df should remain a separate child dataset rather than being flattened directly into the main Item Details table.

### 2. Parent Integrity

| Validation | Result |
|---|---:|
| Tax parent IDs | 77,634 |
| Matched to current Item Details | 77,634 |
| Not found | 0 |

Every tax parent ID checked exists in the current Item Details dataset.

The parent count can differ from the earlier 72,784 cardinality snapshot because the raw GCS data is continuously changing. The profiling is intentionally performed against the fresh current dataset, so the latest validation should be treated as authoritative for this profiling run.

### 3. Completeness

| Field / Group | Finding |
|---|---:|
| `_dlt_id` | 100% |
| `_dlt_list_idx` | 100% |
| `_dlt_parent_id` | 100% |
| `ebay_collect_and_remit_tax` | 100% |
| `included_in_price` | 100% |
| `tax_type` | 100% |
| `Jurisdiction fields` | ~99.93% overall |
| `shipping_and_handling_taxed` | ~99.93% |
| `tax_percentage` | ~0.1% |


The major finding is `tax_percentage` sparsity.

The field is available for only a very small fraction of tax records. However, the populated values themselves pass the validity checks described below.

Therefore:
 Low completeness does not mean bad data.

The field is mostly unavailable from the source, but when supplied, the values appear valid.

For Silver, the correct approach is to preserve the NULLs rather than impute them.

### 4. `tax_percentage` Quality

| Validation | Result |
|---|---:|
| Records with a value | 3,615 |
| Non-numeric values | 0 |
| Negative values | 0 |
| Values above 100% | 0 |
| Observed range | 0–37% |

The populated tax_percentage values are therefore valid according to the current profiling rules.

This gives us an important distinction:
```text
Completeness:
~0.1% populated
        ↓
Very sparse

Validity:
0 non-numeric
0 negative
0 > 100%
        ↓
Valid when present
```

For the Silver layer:
```text
tax_percentage
    → cast to numeric
    → preserve NULLs
    → do NOT impute missing values

The absence of a tax percentage should be treated as source-level unavailability, not automatically as a zero tax rate.
```

### 5. Tax Type Distribution

The current dataset contains two observed tax types:

- STATE_SALES_TAX
- VAT

No unexpected tax types were observed.

### Tax Type vs Jurisdiction Fields

| Tax Type | Records | Region Name NULL | Region Type NULL | Jurisdiction ID NULL |
|---|---:|---:|---:|---:|
| `STATE_SALES_TAX` | 3,649,892 | 0 | 0 | 0 |
| `VAT` | 2,521 | 2,521 | 2,521 | 2,521 |


The important finding is that jurisdiction-field sparsity is strongly associated with tax type.

This is not random missingness.

The correct Silver quality rule should therefore be conditional:
```text
STATE_SALES_TAX
    → jurisdiction information expected

VAT
    → jurisdiction fields may be NULL
```

We should not create a generic rule such as:

**"jurisdiction fields must never be NULL"**

because that would incorrectly classify valid VAT records as bad data.

### 6. Blank-String Quality

The profiling found:
**Empty / whitespace-only strings: 0**


There is no evidence of hidden blank-string values in the tax dataset.

Therefore, no additional blank-string cleanup is required beyond the normal Silver transformation rules.

### Engineering Conclusions


- Bronze

Preserve the source data exactly as received.
```text
Raw tax records
    ↓
Bronze
    ↓
No aggressive correction
```

The Bronze layer should maintain the original NULLs and source representations so that the pipeline remains fully traceable.


- Silver

Apply type normalization and conditional quality rules.
```text
tax_percentage
    → numeric / decimal
    → nullable

tax_type
    → categorical string

jurisdiction fields
    → nullable where source semantics allow

business flags
    → appropriate boolean / categorical types
```

Legitimate source NULLs should remain NULL rather than being replaced with fabricated values.

- Gold

Only derive tax metrics when sufficient source information exists.

In particular, we should not calculate an estimated tax amount for every listing from tax_percentage, because approximately 99.9% of tax records do not provide a tax percentage.

A derived tax metric should therefore be conditional:
```text
IF tax_percentage IS NOT NULL
    → tax-derived metric may be calculated
ELSE
    → metric remains NULL / unavailable
```